In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from custom_layers import Sampling
from tensorflow.keras.layers import (
    Input, Conv3D, AveragePooling3D, BatchNormalization, Add, Activation,
    Flatten, Dense, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from glob import glob
import os
import re
import numpy as np
from skimage.transform import resize
from skimage import io as skio
from tqdm import tqdm

In [ ]:
def preprocess_and_save_with_structure_serial(root_input_dir, root_output_dir, image_paths, bbox_dims=(10, 50, 50), twochan=False):
    '''
    twochan can be provided as a root_input_dir equivalent variable. 
    If it is given as such, the resulting output will be preprocessed 2 channel images of shape z,y,x,c,0 (there needs to be an extra dim appended)
    '''
    def _normalize(image):
        return image / np.max(image)

    def _resize(image):
        scaling_factor = min(1.0, min((bbox_dims[i] / image.shape[i]) for i in range(3)))
        target_dims = [int(image.shape[i] * scaling_factor) for i in range(3)]
        return resize(image, target_dims, mode='reflect', anti_aliasing=True)

    def _pad(image):
        pad_depth = max(0, bbox_dims[0] - image.shape[0])
        pad_height = max(0, bbox_dims[1] - image.shape[1])
        pad_width = max(0, bbox_dims[2] - image.shape[2])

        pad_depth_before, pad_depth_after = pad_depth // 2, pad_depth - pad_depth // 2
        pad_height_before, pad_height_after = pad_height // 2, pad_height - pad_height // 2
        pad_width_before, pad_width_after = pad_width // 2, pad_width - pad_width // 2

        return np.pad(image,
                      ((pad_depth_before, pad_depth_after),
                       (pad_height_before, pad_height_after),
                       (pad_width_before, pad_width_after)),
                      'constant', constant_values=0)

    def convert_to_npy_filename(filename: str) -> str:
        return re.sub(r'(\.tiff*|\.tif)$', '.npy', filename, flags=re.IGNORECASE)

    def _process_and_save_image(full_path):
        try:
            image = skio.imread(full_path)
            image = _normalize(image)
            image = _resize(image)
            image = _pad(image)
            if twochan:
                alt_path = os.path.join(twochan, os.path.relpath(full_path, root_input_dir))
                alt_im = skio.imread(alt_path)
                alt_im = _normalize(alt_im)
                alt_im = _resize(alt_im)
                alt_im = _pad(alt_im)
                image = np.stack([image,alt_im],axis=-1)
            image = np.expand_dims(image, axis=-1).astype(np.float32)

            rel_path = os.path.relpath(full_path, root_input_dir)
            rel_path_no_ext, _ = os.path.splitext(rel_path)
            new_filename = convert_to_npy_filename(rel_path_no_ext)
            save_path = os.path.join(root_output_dir, new_filename + ".npy")
            os.makedirs(os.path.dirname(save_path), exist_ok=True)

            np.save(save_path, image)
            return True
        except Exception as e:
            print(f"[Error] {full_path}: {e}")
            return False

    print(f"Found {len(image_paths)} images. Starting serial preprocessing...")

    for path in tqdm(image_paths, desc="Preprocessing images"):
        _process_and_save_image(path)


In [ ]:
rootdir = '/net/bmc-lab8/data/lab/boyer/users/jhday/Sanofi_Data/5hr_experiments/Lamp1/PostEx/20250921_H9E1_PostEx__2025-09-21T11_24_05-Measurement 1b/Stardist_SegData_combined_c4'

image_paths = glob(f"{rootdir}/**/*.tiff", recursive=True)

In [ ]:
rootoutdir = '/net/bmc-lab8/data/lab/boyer/users/jhday/Sanofi_Data/5hr_experiments/Lamp1/PostEx/20250921_H9E1_PostEx__2025-09-21T11_24_05-Measurement 1b/Stardist_SegData_combined_c3_c4_preprocessed'
seconddir = '/net/bmc-lab8/data/lab/boyer/users/jhday/Sanofi_Data/5hr_experiments/Lamp1/PostEx/20250921_H9E1_PostEx__2025-09-21T11_24_05-Measurement 1b/Stardist_SegData_combined_c2'
preprocess_and_save_with_structure_serial(rootdir,rootoutdir,image_paths,twochan=seconddir)

Decoding

In [3]:
from tensorflow.keras.layers import Reshape, UpSampling3D, Conv3DTranspose
    
def residual_block_decoder2(x, filters, kernel_size=(3, 3, 3), padding='same', strides=(1, 1, 1), l2_lambda=0.01, ACTV='relu'):
    regularization = l2(l2_lambda)
    # Shortcut
    shortcut = Conv3D(filters, kernel_size=(1, 1, 1), activation=ACTV, strides=strides, padding=padding, kernel_regularizer=regularization)(x)
    shortcut = UpSampling3D((2, 2, 2))(shortcut)
    # First Conv3D
    x = Conv3DTranspose(filters, kernel_size=(4,4,4), activation=ACTV, padding=padding, strides=(2,2,2), kernel_regularizer=regularization)(x)
    x = BatchNormalization()(x)
    # Second Conv3D
    x = Conv3D(filters, kernel_size=(3,3,3), activation=ACTV, padding=padding, strides=strides, kernel_regularizer=regularization)(x)
    x = BatchNormalization()(x)

    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

# Define the decoder part for 3D input
def decoder(encoded, L2_lambda,
            res_block_filters=[64, 32, 16, 8], # also changed
            dropout_rate=0.0,
            reshape_dims=(1, 4, 4)): # changed
    num_channels = res_block_filters[0]
    dense_units = reshape_dims[0] * reshape_dims[1] * reshape_dims[2] * num_channels
    x = Dense(dense_units, activation='relu')(encoded)
    x = Reshape((*reshape_dims, num_channels))(x)

    for filters in res_block_filters:
        x = residual_block_decoder2(x, filters, l2_lambda=L2_lambda)
        if dropout_rate > 0:
            x = tf.keras.layers.Dropout(dropout_rate)(x)

    decoded = Conv3D(
        2, kernel_size=(3, 3, 3), activation='sigmoid', # changed to 2 channels
        padding='same', strides=(1, 1, 1),
        kernel_regularizer=l2(L2_lambda)
    )(x)

    return decoded

In [4]:
inputs = Input(shape=(256,))
image = decoder(inputs, L2_lambda=0.01)
decoder_model = Model(inputs, image)
decoder_model.load_weights("fortunate_shrike_decoder.h5")

2026-06-14 15:20:11.785564: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'fortunate_shrike_decoder.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [5]:
import pandas as pd
import numpy as np
import tifffile as tif
def cluster_mean_vectors(
    df,
    cluster_col,
    feature_prefix = "z",
    feature_cols = None,
):
    """
    Generate one mean feature vector per cluster.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame where rows are objects.
    cluster_col : str
        Name of the column containing cluster IDs.
    feature_prefix : str
        Prefix used to identify feature columns, default assumes z000, z001, etc.
    feature_cols : list[str] | None
        Optional explicit list of feature columns. If None, inferred from prefix.

    Returns
    -------
    pd.DataFrame
        One row per cluster, with averaged feature values.
    """

    if cluster_col not in df.columns:
        raise ValueError(f"{cluster_col!r} is not a column in df.")

    if feature_cols is None:
        feature_cols = sorted(
            [
                col for col in df.columns
                if col.startswith(feature_prefix) and col[len(feature_prefix):].isdigit()
            ],
            key=lambda x: int(x[len(feature_prefix):])
        )

    if len(feature_cols) == 0:
        raise ValueError("No feature columns found.")

    missing = [col for col in feature_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Feature columns missing from df: {missing}")

    mean_df = (
        df
        .groupby(cluster_col, dropna=False)[feature_cols]
        .mean()
        .reset_index()
    )

    return mean_df

In [6]:
df = pd.read_csv('/net/bmc-lab8/data/lab/boyer/users/jhday/Sanofi Paper/June2026_Submission/data/Figure3_4_Data.csv')

Decoding an example vector

In [ ]:
outdir = "/net/bmc-lab8/data/lab/boyer/users/jhday/Sanofi_Data/Compound_Panel_Experiments/Decoding"
deepfeatures = [f"z{i:03d}" for i in range(0, 256)]
vector = deepclust.loc[deepclust.index[13263], deepfeatures].to_numpy(dtype=np.float32)
X = vector[None, :]  # shape (1, 256)
# decode
pred = decoder_model.predict(X)
pred = np.squeeze(pred)  # expected shape (z, y, x, 2)
lamp1 = pred[:, :, :, 0]
lamp3 = pred[:, :, :, 1]
tif.imwrite(f"{outdir}/000_im_lamp1_example.tiff", lamp1)
tif.imwrite(f"{outdir}/000_im_lamp3_example.tiff", lamp3)

Decoding cluster mean vectors

In [ ]:
cluster_vectors = cluster_mean_vectors(df,cluster_col="Learned Feature Cluster")
deepfeatures = [f"z{i:03d}" for i in range(0, 256)]
outdir = "/net/bmc-lab8/data/lab/boyer/users/jhday/Sanofi_Data/Compound_Panel_Experiments/Decoding/Leiden_clusters"
df=cluster_vectors
for i in range(len(df)):
    cluster = df["leiden_res_1.5"].iloc[i]
    # get the latent vector for this row
    vector = df.loc[df.index[i], deepfeatures].to_numpy(dtype=np.float32)
    X = vector[None, :]  # shape (1, 256)
    # decode
    pred = decoder_model.predict(X)
    pred = np.squeeze(pred)  # expected shape (z, y, x, 2)
    lamp1 = pred[:, :, :, 0]
    lamp3 = pred[:, :, :, 1]
    tif.imwrite(f"{outdir}/000_im_lamp1_cluster{cluster}_row.tiff", lamp1)
    tif.imwrite(f"{outdir}/000_im_lamp3_cluster{cluster}_row.tiff", lamp3)